In [1]:
print("najla")

najla


In [2]:
# Import operating-system utilities for environment handling.
import os

# Import system utilities for modifying Python import paths.
import sys

# Import random to control Python-level randomness.
import random

# Import copy to save the best model state during early stopping.
import copy

# Import Path to write cleaner file paths.
from pathlib import Path

# Import NumPy for numerical arrays.
import numpy as np

# Import pandas for tabular data handling.
import pandas as pd

# Import PyTorch.
import torch

# Import PyTorch neural-network modules.
import torch.nn as nn

# Import PyTorch functional operations such as dropout and activation.
import torch.nn.functional as F

# Import ColumnTransformer to combine numeric and categorical preprocessing.
from sklearn.compose import ColumnTransformer

# Import SimpleImputer to fill missing numeric/categorical values safely.
from sklearn.impute import SimpleImputer

# Import Pipeline to chain imputation and scaling/encoding.
from sklearn.pipeline import Pipeline

# Import StandardScaler to scale numerical features.
from sklearn.preprocessing import StandardScaler

# Import OneHotEncoder to convert categorical variables into numeric columns.
from sklearn.preprocessing import OneHotEncoder

from sklearn.decomposition import PCA

from sentence_transformers import SentenceTransformer

# Import W&B for experiment tracking.
import wandb

# Import GATConv, the Graph Attention Network layer from PyTorch Geometric.
from torch_geometric.nn import GATConv

# Import display to inspect DataFrames inside the notebook.
from IPython.display import display

# Define your project root.
project_root = Path("/home/najla/dev/najla-msc/bikeshare/")

# Add the project root to Python path if it is not already there.
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import shared model helper from your project package.
from scripts.model_helper_stations_lvl import BikeShareModelHelper as mh


In [3]:
# Define the directory containing the Toronto graph files.
graph_dir = Path("/home/najla/dev/najla-msc/bikeshare/data/processed/graph")

# Read graph edges and reset index so level_0 and level_1 become columns.
edges = pd.read_parquet(graph_dir / "edges.parquet").reset_index()

edges

,level_0,level_1,weight,geometry
0,5320100.01,5350805.21,5229.881836,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x17\x9c...
1,5320100.01,5320100.02,2923.277907,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x17\x9c...
2,5320100.01,5350805.04,4516.007185,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x17\x9c...
3,5320100.02,5350805.21,2780.244458,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\xa8\x05...
4,5320100.02,5350805.23,3346.251980,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\xa8\x05...
...,...,...,...,...
4085,5680100.00,5680101.00,8313.999958,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\xd4\xf8...
4086,5680100.00,5680103.08,6211.882070,"b""\x01\x02\x00\x00\x00\x02\x00\x00\x00\xd4\xf8..."
4087,5680102.01,5680104.00,3830.837493,b'\x01\x02\x00\x00\x00\x02\x00\x00\x005N\xe4\x...
4088,5680103.05,5680104.00,2911.674860,b'\x01\x02\x00\x00\x00\x02\x00\x00\x00\x14\xce...


In [4]:
# Define the directory containing the processed modelling dataframe.
data_dir = Path("/home/najla/dev/najla-msc/bikeshare/data")

# Read the same processed CT-date dataframe used by MLP.
df_txt_node2vec  = pd.read_parquet(data_dir / "processed/model_df/df_2targets_txtTokens_graphFeat.parquet")

df_txt_node2vec.head(10)

,loc_id,loc_name,lon,lat,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,...,node2vec_12,node2vec_13,node2vec_14,node2vec_15,day_type,public_holiday,Main_Weather_Category,season,spatial_group,attraction_missing_flag
0,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
1,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
2,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
3,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
4,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,new_years_day,snowy,winter,19,0
5,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,no_public_holiday,snowy,winter,19,0
6,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekend,no_public_holiday,snowy,winter,19,0
7,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekday,no_public_holiday,very_cold,winter,19,0
8,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekday,no_public_holiday,very_cold,winter,19,0
9,5350001.00,0001.00,-79.334142,43.646019,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,...,1.38141,-2.212004,1.429404,1.111674,weekday,no_public_holiday,very_cold,winter,19,0


In [5]:
# labelled dataset
df_txt_node2vec.columns

Index(['loc_id', 'loc_name', 'lon', 'lat', 'population', 'jobs', 'area',
       'poi_arts_and_entertainment', 'poi_community_and_government',
       'poi_cultural_and_historic', 'poi_education', 'poi_food_and_drink',
       'poi_geographic_entities', 'poi_health_care', 'poi_lifestyle_services',
       'poi_lodging', 'poi_services_and_business', 'poi_shopping',
       'poi_sports_and_recreation', 'poi_travel_and_transportation',
       'land_use_agriculture', 'land_use_campground', 'land_use_cemetery',
       'land_use_construction', 'land_use_developed', 'land_use_education',
       'land_use_entertainment', 'land_use_golf', 'land_use_horticulture',
       'land_use_landfill', 'land_use_managed', 'land_use_medical',
       'land_use_military', 'land_use_park', 'land_use_pedestrian',
       'land_use_protected', 'land_use_recreation', 'land_use_religious',
       'land_use_residential', 'land_use_resource_extraction',
       'land_use_transportation', 'land_use_winter_sports',
       's

In [6]:
# Count distinct loc_id values in df_txt_node2vec.
df_txt_node2vec["inflow_count"].isna().sum()

np.int64(0)

Add the rest of the nodes to the graph df to make full graph

In [7]:
df_txt_node2vec = df_txt_node2vec.drop(columns=['node2vec_0', 'node2vec_1', 'node2vec_2', \
       'node2vec_3', 'node2vec_4', 'node2vec_5', 'node2vec_6', 'node2vec_7', \
       'node2vec_8', 'node2vec_9', 'node2vec_10', 'node2vec_11', 'node2vec_12', \
       'node2vec_13', 'node2vec_14', 'node2vec_15', 'lat', 'lon'])
# Convert date column to datetime.
df_txt_node2vec["date"] = pd.to_datetime(df_txt_node2vec["date"])
df_txt_node2vec.head() #drop lat, lon, loc_id_key, end_loc_id_key, columns start with node

,loc_id,loc_name,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,poi_education,poi_food_and_drink,...,inflow_count,outflow_count,attraction_count,wiki_items_text,day_type,public_holiday,Main_Weather_Category,season,spatial_group,attraction_missing_flag
0,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,2.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
1,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,2.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
2,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,1.0,0.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
3,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
4,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.0,3.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0


In [8]:
df_txt_node2vec.columns

Index(['loc_id', 'loc_name', 'population', 'jobs', 'area',
       'poi_arts_and_entertainment', 'poi_community_and_government',
       'poi_cultural_and_historic', 'poi_education', 'poi_food_and_drink',
       'poi_geographic_entities', 'poi_health_care', 'poi_lifestyle_services',
       'poi_lodging', 'poi_services_and_business', 'poi_shopping',
       'poi_sports_and_recreation', 'poi_travel_and_transportation',
       'land_use_agriculture', 'land_use_campground', 'land_use_cemetery',
       'land_use_construction', 'land_use_developed', 'land_use_education',
       'land_use_entertainment', 'land_use_golf', 'land_use_horticulture',
       'land_use_landfill', 'land_use_managed', 'land_use_medical',
       'land_use_military', 'land_use_park', 'land_use_pedestrian',
       'land_use_protected', 'land_use_recreation', 'land_use_religious',
       'land_use_residential', 'land_use_resource_extraction',
       'land_use_transportation', 'land_use_winter_sports',
       'stations_capaci

The columns with empty values in the rest of nodes attraction_count, wiki_items_text, spatial_group, trip_count
To add the rest of the nodes:
1. df_temporal_features = select end_date, day_type, public_holiday Main_Weather_Category, season
2. get nodes id not in end_loc_id and cross join them with df_temporal_features
3. join them back to nodes features in project_root = Path("/home/najla/dev/najla-msc")
graph_dir = project_root / "data/processed/THATS/NetworkGraph"
4. stack them over the current df

In [9]:
# read all nodes
project_root = Path("/home/najla/dev/najla-msc")
graph_dir = project_root / "bikeshare/data/processed/graph"

# Replace nodes.parquet if your node-feature file has another name.
all_node_features = pd.read_parquet(
    graph_dir / "nodes.parquet"
).drop(columns=["loc_name","type","lon","lat","geometry","original_geometry"]).reset_index()
all_node_features 

,loc_id,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,poi_education,poi_food_and_drink,poi_geographic_entities,...,land_use_military,land_use_park,land_use_pedestrian,land_use_protected,land_use_recreation,land_use_religious,land_use_residential,land_use_resource_extraction,land_use_transportation,land_use_winter_sports
0,5320100.01,0.000451,0.000203,16220000.0,4.637299e-07,1.391190e-06,6.376286e-07,4.057637e-07,1.565088e-06,4.637299e-07,...,0.0,0.003531,0.000044,0.003211,0.000218,0.000000,0.012452,0.000000,0.001572,0.0
1,5320100.02,0.002052,0.000875,2680000.0,3.708509e-07,3.708509e-07,3.708509e-07,3.708509e-06,5.191913e-06,0.000000e+00,...,0.0,0.010381,0.000000,0.000000,0.000587,0.000000,0.134664,0.000000,0.000000,0.0
2,5320100.03,0.001365,0.000666,5940000.0,3.367461e-07,3.367461e-07,6.734923e-07,1.178611e-06,1.346985e-06,0.000000e+00,...,0.0,0.014758,0.000000,0.000000,0.000423,0.002103,0.041123,0.000000,0.000000,0.0
3,5320105.14,0.000334,0.000157,15920000.0,1.250645e-07,3.126613e-07,1.250645e-07,5.627904e-07,2.751420e-06,0.000000e+00,...,0.0,0.003706,0.000066,0.000000,0.000069,0.000000,0.017175,0.024266,0.000000,0.0
4,5320105.17,0.000032,0.000016,52030000.0,3.834821e-08,1.342188e-07,1.150446e-07,1.917411e-08,3.834821e-08,0.000000e+00,...,0.0,0.000096,0.000000,0.000000,0.000019,0.001239,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,5680101.00,0.000033,0.000016,170910000.0,4.091783e-08,8.183566e-08,6.429944e-08,8.183566e-08,3.214972e-07,1.169081e-08,...,0.0,0.000082,0.000000,0.000000,0.000040,0.000013,0.000685,0.000000,0.000000,0.0
1244,5680102.01,0.000161,0.000076,19120000.0,2.431003e-08,4.862005e-08,2.431003e-08,0.000000e+00,3.160303e-07,9.724011e-08,...,0.0,0.000407,0.000027,0.025996,0.000167,0.000000,0.005397,0.000000,0.000000,0.0
1245,5680103.05,0.001307,0.000642,3170000.0,3.493472e-07,2.328981e-07,0.000000e+00,0.000000e+00,1.513838e-06,8.151434e-07,...,0.0,0.011240,0.000000,0.000000,0.001347,0.000000,0.066406,0.000000,0.000000,0.0
1246,5680103.08,0.001540,0.000703,2410000.0,0.000000e+00,2.554194e-07,0.000000e+00,2.554194e-07,8.939678e-07,6.385484e-07,...,0.0,0.002475,0.000000,0.000000,0.000315,0.000000,0.132143,0.000000,0.000000,0.0


In [10]:
# make sure the data types of the keys are the same
df_txt_node2vec["loc_id"] = (
    pd.to_numeric(
        df_txt_node2vec["loc_id"],
        errors="raise",
    )
    .map(lambda value: f"{value:.2f}")
)

all_node_features["loc_id"] = (
    pd.to_numeric(
        all_node_features["loc_id"],
        errors="raise",
    )
    .map(lambda value: f"{value:.2f}")
)
all_node_features

,loc_id,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,poi_education,poi_food_and_drink,poi_geographic_entities,...,land_use_military,land_use_park,land_use_pedestrian,land_use_protected,land_use_recreation,land_use_religious,land_use_residential,land_use_resource_extraction,land_use_transportation,land_use_winter_sports
0,5320100.01,0.000451,0.000203,16220000.0,4.637299e-07,1.391190e-06,6.376286e-07,4.057637e-07,1.565088e-06,4.637299e-07,...,0.0,0.003531,0.000044,0.003211,0.000218,0.000000,0.012452,0.000000,0.001572,0.0
1,5320100.02,0.002052,0.000875,2680000.0,3.708509e-07,3.708509e-07,3.708509e-07,3.708509e-06,5.191913e-06,0.000000e+00,...,0.0,0.010381,0.000000,0.000000,0.000587,0.000000,0.134664,0.000000,0.000000,0.0
2,5320100.03,0.001365,0.000666,5940000.0,3.367461e-07,3.367461e-07,6.734923e-07,1.178611e-06,1.346985e-06,0.000000e+00,...,0.0,0.014758,0.000000,0.000000,0.000423,0.002103,0.041123,0.000000,0.000000,0.0
3,5320105.14,0.000334,0.000157,15920000.0,1.250645e-07,3.126613e-07,1.250645e-07,5.627904e-07,2.751420e-06,0.000000e+00,...,0.0,0.003706,0.000066,0.000000,0.000069,0.000000,0.017175,0.024266,0.000000,0.0
4,5320105.17,0.000032,0.000016,52030000.0,3.834821e-08,1.342188e-07,1.150446e-07,1.917411e-08,3.834821e-08,0.000000e+00,...,0.0,0.000096,0.000000,0.000000,0.000019,0.001239,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,5680101.00,0.000033,0.000016,170910000.0,4.091783e-08,8.183566e-08,6.429944e-08,8.183566e-08,3.214972e-07,1.169081e-08,...,0.0,0.000082,0.000000,0.000000,0.000040,0.000013,0.000685,0.000000,0.000000,0.0
1244,5680102.01,0.000161,0.000076,19120000.0,2.431003e-08,4.862005e-08,2.431003e-08,0.000000e+00,3.160303e-07,9.724011e-08,...,0.0,0.000407,0.000027,0.025996,0.000167,0.000000,0.005397,0.000000,0.000000,0.0
1245,5680103.05,0.001307,0.000642,3170000.0,3.493472e-07,2.328981e-07,0.000000e+00,0.000000e+00,1.513838e-06,8.151434e-07,...,0.0,0.011240,0.000000,0.000000,0.001347,0.000000,0.066406,0.000000,0.000000,0.0
1246,5680103.08,0.001540,0.000703,2410000.0,0.000000e+00,2.554194e-07,0.000000e+00,2.554194e-07,8.939678e-07,6.385484e-07,...,0.0,0.002475,0.000000,0.000000,0.000315,0.000000,0.132143,0.000000,0.000000,0.0


In [11]:
# create the temporal df to cross join it later with the node features
temporal_cols = [
    "date",
    "day_type",
    "public_holiday",
    "Main_Weather_Category",
    "season",
]

df_temporal_features = (
    df_txt_node2vec[temporal_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_temporal_features

,date,day_type,public_holiday,Main_Weather_Category,season
0,2022-01-01,weekend,new_years_day,snowy,winter
1,2022-01-02,weekend,no_public_holiday,snowy,winter
2,2022-01-03,weekday,no_public_holiday,very_cold,winter
3,2022-01-04,weekday,no_public_holiday,very_cold,winter
4,2022-01-05,weekday,no_public_holiday,windy,winter
...,...,...,...,...,...
1546,2022-12-24,weekend,no_public_holiday,snowy,winter
1547,2022-12-25,weekend,christmas_day,snowy,winter
1548,2025-02-16,weekend,no_public_holiday,snowy,winter
1549,2025-02-17,weekday,family_day,snowy,winter


In [12]:
# Remove the island CT before building the full CT-date panel.
BROKEN_ISLAND_LOC_ID = "5350002.00"

df_txt_node2vec = (
    df_txt_node2vec
    .loc[~df_txt_node2vec["loc_id"].eq(BROKEN_ISLAND_LOC_ID)]
    .reset_index(drop=True)
)

all_node_features = (
    all_node_features
    .loc[~all_node_features["loc_id"].eq(BROKEN_ISLAND_LOC_ID)]
    .reset_index(drop=True)
)

# Build every possible CT-date pair from the fixed CT node table and the study dates.
# This creates the full graph node panel for every date.
all_node_date_df = all_node_features.merge(
    df_temporal_features,
    how="cross",
)

# Keep the CT-date pairs that already have labelled station rows.
# These labelled rows have priority and should not be replaced by filler rows.
labelled_node_date_keys = (
    df_txt_node2vec[["loc_id", "date"]]
    .drop_duplicates()
    .assign(has_labelled_node_date=1)
)


# Add only CT-date pairs that are missing from the labelled station data.
# This fixes the bug: the filter is now loc_id + date, not loc_id alone.
missing_node_date_df = (
    all_node_date_df
    .merge(
        labelled_node_date_keys,
        on=["loc_id", "date"],
        how="left",
    )
    .loc[lambda df: df["has_labelled_node_date"].isna()]
    .drop(columns=["has_labelled_node_date"])
    .reset_index(drop=True)
)

In [13]:
error

NameError: name 'error' is not defined

In [14]:
# Add the missing columns
# These nodes do not have observed cyclist inflow.
missing_node_date_df["inflow_count"] = np.nan
missing_node_date_df["outflow_count"] = np.nan

# These nodes are not part of the labelled spatial split.
missing_node_date_df["spatial_group"] = np.nan


missing_node_date_df["wiki_items_text"] = "no_wikidata"
missing_node_date_df["attraction_missing_flag"] = 1
missing_node_date_df["attraction_count"] = -1

missing_node_date_df


,loc_id,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,poi_education,poi_food_and_drink,poi_geographic_entities,...,day_type,public_holiday,Main_Weather_Category,season,inflow_count,outflow_count,spatial_group,wiki_items_text,attraction_missing_flag,attraction_count
0,5320100.01,0.000451,0.000203,16220000.0,4.637299e-07,0.000001,6.376286e-07,4.057637e-07,1.565088e-06,4.637299e-07,...,weekend,new_years_day,snowy,winter,NaN,NaN,NaN,no_wikidata,1,-1
1,5320100.01,0.000451,0.000203,16220000.0,4.637299e-07,0.000001,6.376286e-07,4.057637e-07,1.565088e-06,4.637299e-07,...,weekend,no_public_holiday,snowy,winter,NaN,NaN,NaN,no_wikidata,1,-1
2,5320100.01,0.000451,0.000203,16220000.0,4.637299e-07,0.000001,6.376286e-07,4.057637e-07,1.565088e-06,4.637299e-07,...,weekday,no_public_holiday,very_cold,winter,NaN,NaN,NaN,no_wikidata,1,-1
3,5320100.01,0.000451,0.000203,16220000.0,4.637299e-07,0.000001,6.376286e-07,4.057637e-07,1.565088e-06,4.637299e-07,...,weekday,no_public_holiday,very_cold,winter,NaN,NaN,NaN,no_wikidata,1,-1
4,5320100.01,0.000451,0.000203,16220000.0,4.637299e-07,0.000001,6.376286e-07,4.057637e-07,1.565088e-06,4.637299e-07,...,weekday,no_public_holiday,windy,winter,NaN,NaN,NaN,no_wikidata,1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1660061,5680104.00,0.000326,0.000115,11830000.0,5.738482e-08,0.000000,0.000000e+00,0.000000e+00,1.147696e-07,5.738482e-08,...,weekend,no_public_holiday,snowy,winter,NaN,NaN,NaN,no_wikidata,1,-1
1660062,5680104.00,0.000326,0.000115,11830000.0,5.738482e-08,0.000000,0.000000e+00,0.000000e+00,1.147696e-07,5.738482e-08,...,weekend,christmas_day,snowy,winter,NaN,NaN,NaN,no_wikidata,1,-1
1660063,5680104.00,0.000326,0.000115,11830000.0,5.738482e-08,0.000000,0.000000e+00,0.000000e+00,1.147696e-07,5.738482e-08,...,weekend,no_public_holiday,snowy,winter,NaN,NaN,NaN,no_wikidata,1,-1
1660064,5680104.00,0.000326,0.000115,11830000.0,5.738482e-08,0.000000,0.000000e+00,0.000000e+00,1.147696e-07,5.738482e-08,...,weekday,family_day,snowy,winter,NaN,NaN,NaN,no_wikidata,1,-1


In [15]:
missing_node_date_df.columns

Index(['loc_id', 'population', 'jobs', 'area', 'poi_arts_and_entertainment',
       'poi_community_and_government', 'poi_cultural_and_historic',
       'poi_education', 'poi_food_and_drink', 'poi_geographic_entities',
       'poi_health_care', 'poi_lifestyle_services', 'poi_lodging',
       'poi_services_and_business', 'poi_shopping',
       'poi_sports_and_recreation', 'poi_travel_and_transportation',
       'land_use_agriculture', 'land_use_campground', 'land_use_cemetery',
       'land_use_construction', 'land_use_developed', 'land_use_education',
       'land_use_entertainment', 'land_use_golf', 'land_use_horticulture',
       'land_use_landfill', 'land_use_managed', 'land_use_medical',
       'land_use_military', 'land_use_park', 'land_use_pedestrian',
       'land_use_protected', 'land_use_recreation', 'land_use_religious',
       'land_use_residential', 'land_use_resource_extraction',
       'land_use_transportation', 'land_use_winter_sports', 'date', 'day_type',
       'public_

In [16]:
# Apply the same missing-value treatment to labelled rows.
df_txt_node2vec["wiki_items_text"] = (
    df_txt_node2vec["wiki_items_text"]
    .fillna("no_wikidata")
    .replace({"no_wiki_data": "no_wikidata"})
)

df_txt_node2vec["attraction_missing_flag"] = (
    df_txt_node2vec["attraction_count"].isna()
    | df_txt_node2vec["attraction_count"].eq(-1)
    | df_txt_node2vec["wiki_items_text"].eq("no_wikidata")
).astype(int)

df_txt_node2vec.loc[
    df_txt_node2vec["attraction_missing_flag"].eq(1),
    "attraction_count",
] = -1

df_txt_node2vec

,loc_id,loc_name,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,poi_education,poi_food_and_drink,...,inflow_count,outflow_count,attraction_count,wiki_items_text,day_type,public_holiday,Main_Weather_Category,season,spatial_group,attraction_missing_flag
0,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,2.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
1,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,2.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
2,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,1.0,0.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
3,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,0.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
4,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,0.0,3.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
740500,5350802.02,0802.02,0.002377,0.001128,2260000.0,4.062791e-07,0.000001,0.000000e+00,1.218837e-06,8.125581e-07,...,2.0,2.0,2.0,neighborhood urban-type_settlement,weekday,no_public_holiday,clear_weather,autumn,14,0
740501,5350802.02,0802.02,0.002377,0.001128,2260000.0,4.062791e-07,0.000001,0.000000e+00,1.218837e-06,8.125581e-07,...,1.0,1.0,2.0,neighborhood urban-type_settlement,weekend,no_public_holiday,clear_weather,autumn,14,0
740502,5350802.02,0802.02,0.002377,0.001128,2260000.0,4.062791e-07,0.000001,0.000000e+00,1.218837e-06,8.125581e-07,...,1.0,1.0,2.0,neighborhood urban-type_settlement,weekday,no_public_holiday,clear_weather,autumn,14,0
740503,5350802.02,0802.02,0.002377,0.001128,2260000.0,4.062791e-07,0.000001,0.000000e+00,1.218837e-06,8.125581e-07,...,1.0,0.0,2.0,neighborhood urban-type_settlement,weekend,no_public_holiday,clear_weather,autumn,14,0


In [17]:
# Create the complete node-date panel
# Stack the additional node-date rows below the current dataframe.
full_node_df = pd.concat(
    [df_txt_node2vec, missing_node_date_df],
    ignore_index=True,
    sort=False,
)

# the full features df
full_node_df

,loc_id,loc_name,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,poi_education,poi_food_and_drink,...,inflow_count,outflow_count,attraction_count,wiki_items_text,day_type,public_holiday,Main_Weather_Category,season,spatial_group,attraction_missing_flag
0,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,2.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19.0,0
1,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,2.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19.0,0
2,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,1.0,0.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19.0,0
3,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,0.0,1.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19.0,0
4,5350001.00,0001.00,0.000088,0.000055,6820000.0,3.780645e-06,0.000002,5.816377e-07,7.270471e-07,4.216873e-06,...,0.0,3.0,13.0,artificial_island bay beach building bus_garag...,weekend,new_years_day,snowy,winter,19.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2400566,5680104.00,NaN,0.000326,0.000115,11830000.0,5.738482e-08,0.000000,0.000000e+00,0.000000e+00,1.147696e-07,...,NaN,NaN,-1.0,no_wikidata,weekend,no_public_holiday,snowy,winter,NaN,1
2400567,5680104.00,NaN,0.000326,0.000115,11830000.0,5.738482e-08,0.000000,0.000000e+00,0.000000e+00,1.147696e-07,...,NaN,NaN,-1.0,no_wikidata,weekend,christmas_day,snowy,winter,NaN,1
2400568,5680104.00,NaN,0.000326,0.000115,11830000.0,5.738482e-08,0.000000,0.000000e+00,0.000000e+00,1.147696e-07,...,NaN,NaN,-1.0,no_wikidata,weekend,no_public_holiday,snowy,winter,NaN,1
2400569,5680104.00,NaN,0.000326,0.000115,11830000.0,5.738482e-08,0.000000,0.000000e+00,0.000000e+00,1.147696e-07,...,NaN,NaN,-1.0,no_wikidata,weekday,family_day,snowy,winter,NaN,1


In [18]:
full_node_df.columns

Index(['loc_id', 'loc_name', 'population', 'jobs', 'area',
       'poi_arts_and_entertainment', 'poi_community_and_government',
       'poi_cultural_and_historic', 'poi_education', 'poi_food_and_drink',
       'poi_geographic_entities', 'poi_health_care', 'poi_lifestyle_services',
       'poi_lodging', 'poi_services_and_business', 'poi_shopping',
       'poi_sports_and_recreation', 'poi_travel_and_transportation',
       'land_use_agriculture', 'land_use_campground', 'land_use_cemetery',
       'land_use_construction', 'land_use_developed', 'land_use_education',
       'land_use_entertainment', 'land_use_golf', 'land_use_horticulture',
       'land_use_landfill', 'land_use_managed', 'land_use_medical',
       'land_use_military', 'land_use_park', 'land_use_pedestrian',
       'land_use_protected', 'land_use_recreation', 'land_use_religious',
       'land_use_residential', 'land_use_resource_extraction',
       'land_use_transportation', 'land_use_winter_sports',
       'stations_capaci

In [19]:
# check out the results
print(
    "Full rows:",
    len(full_node_df),
)

print(
    "Full nodes:",
    full_node_df["loc_id"].nunique(),
)

print(
    "Dates:",
    full_node_df["date"].nunique(),
)

print(
    "Rows with inflow counts:",
    full_node_df["inflow_count"].notna().sum(),
)

print(
    "Rows without outflow counts:",
    full_node_df["outflow_count"].isna().sum(),
)

Full rows: 2400571
Full nodes: 1247
Dates: 1551
Rows with inflow counts: 740505
Rows without outflow counts: 1660066


In [20]:
error

NameError: name 'error' is not defined

Data Transform

In [21]:

# =========================================================
# Read graph-ready categorised Wikidata tokens
# =========================================================

# This file includes both real Wikidata CTs and graph-only no_wikidata CTs.
cat_all_wikidata_tokens = pd.read_parquet(
    graph_dir / "graph_cat_all_wikidata_tokens.parquet"
)
# Keep one categorised Wikidata-token row per CT.
cat_wikidata_tokens_clean = (
    cat_all_wikidata_tokens[
        [
            "loc_id",
            "wiki_items_text",
            "attraction_count",
            "attraction_missing_flag",
        ]
    ]
    .drop_duplicates(subset=["loc_id"])
    .rename(columns={"attraction_count": "token_attraction_count"})
    .reset_index(drop=True)
)

# Drop old token-derived columns before joining the fresh categorised tokens.
full_node_df = full_node_df.drop(
    columns=[
        "wiki_items_text",
        "token_attraction_count",
        "attraction_missing_flag",
    ],
    errors="ignore",
)

# Left join keeps every graph node and adds the categorised token fields.
full_node_df = full_node_df.merge(
    cat_wikidata_tokens_clean,
    on="loc_id",
    how="left",
)

# Keep the final loc_id-level text table for embedding.
full_node_wiki = (
    full_node_df[["loc_id", "wiki_items_text"]]
    .drop_duplicates(subset=["loc_id"])
    .reset_index(drop=True)
)

########################## minilm embedding for the full node df

# Load MiniLM.
text_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
    )


# produce miniLLM
# Encode every CT in the full Wikidata table.
wiki_embeddings = text_model.encode(
    full_node_wiki["wiki_items_text"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    )

# Name the embedding columns.
wiki_cols = [
    f"wiki_emb_{i}"
    for i in range(wiki_embeddings.shape[1])
    ]

# Create the saved embedding table.
wiki_embedding_df = pd.concat(
        [
            full_node_wiki[["loc_id"]].reset_index(drop=True),
            pd.DataFrame(
                wiki_embeddings,
                columns=wiki_cols,
            ),
        ],
        axis=1,
    )


wiki_embedding_df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/39 [00:00<?, ?it/s]

,loc_id,wiki_emb_0,wiki_emb_1,wiki_emb_2,wiki_emb_3,wiki_emb_4,wiki_emb_5,wiki_emb_6,wiki_emb_7,wiki_emb_8,...,wiki_emb_374,wiki_emb_375,wiki_emb_376,wiki_emb_377,wiki_emb_378,wiki_emb_379,wiki_emb_380,wiki_emb_381,wiki_emb_382,wiki_emb_383
0,5350001.00,0.122401,-0.048670,0.080268,0.035624,-0.014239,0.046509,0.021445,-0.037985,-0.129345,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
1,5350003.00,0.099267,0.005671,0.044469,-0.001131,-0.009976,0.069001,-0.016951,-0.021439,-0.084655,...,0.009194,0.040855,0.050107,0.000374,0.045037,-0.024544,0.118833,0.044494,-0.011805,0.052420
2,5350004.00,0.100787,0.019868,0.040966,0.011037,-0.026639,0.066037,-0.019478,-0.075967,-0.078240,...,0.010068,-0.019136,0.061460,0.058281,0.081440,-0.032126,0.144641,0.067130,0.028224,0.024335
3,5350005.00,0.114207,-0.005456,0.019324,0.024685,-0.016106,0.081657,-0.006356,-0.103705,-0.079513,...,0.004057,-0.031793,0.037107,0.043067,0.055388,-0.082705,0.111031,0.049507,0.020886,0.036475
4,5350007.01,-0.008696,0.049782,-0.057585,-0.023322,-0.022540,-0.093846,0.073936,-0.013425,0.025058,...,-0.035221,-0.013174,0.062141,0.103825,-0.104846,-0.047563,0.120084,-0.038118,0.015093,-0.033621
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1242,5680101.00,-0.008696,0.049782,-0.057585,-0.023322,-0.022540,-0.093846,0.073936,-0.013425,0.025058,...,-0.035221,-0.013174,0.062141,0.103825,-0.104846,-0.047563,0.120084,-0.038118,0.015093,-0.033621
1243,5680102.01,-0.008696,0.049782,-0.057585,-0.023322,-0.022540,-0.093846,0.073936,-0.013425,0.025058,...,-0.035221,-0.013174,0.062141,0.103825,-0.104846,-0.047563,0.120084,-0.038118,0.015093,-0.033621
1244,5680103.05,-0.008696,0.049782,-0.057585,-0.023322,-0.022540,-0.093846,0.073936,-0.013425,0.025058,...,-0.035221,-0.013174,0.062141,0.103825,-0.104846,-0.047563,0.120084,-0.038118,0.015093,-0.033621
1245,5680103.08,-0.008696,0.049782,-0.057585,-0.023322,-0.022540,-0.093846,0.073936,-0.013425,0.025058,...,-0.035221,-0.013174,0.062141,0.103825,-0.104846,-0.047563,0.120084,-0.038118,0.015093,-0.033621


In [22]:
# Join the MiniLM embeddings to the full graph dataframe.
full_node_df = full_node_df.merge(
    wiki_embedding_df,
    on="loc_id",
    how="left",
)

# Sanity check before dropping wiki_items_text.
no_wikidata_nodes = set(
    full_node_df.loc[
        full_node_df["wiki_items_text"].eq("no_wikidata"),
        "loc_id",
    ].unique()
)

missing_count_nodes = set(
    full_node_df.loc[
        full_node_df["attraction_count"].eq(-1),
        "loc_id",
    ].unique()
)

missing_flag_nodes = set(
    full_node_df.loc[
        full_node_df["attraction_missing_flag"].eq(1),
        "loc_id",
    ].unique()
)

print("Nodes with no Wikidata:", len(no_wikidata_nodes))
print("Nodes with attraction_count = -1:", len(missing_count_nodes))
print("Nodes with attraction_missing_flag = 1:", len(missing_flag_nodes))
print(
    "Same node sets:",
    no_wikidata_nodes == missing_count_nodes == missing_flag_nodes,
)


# Remove the original text column from the saved MiniLM graph features.
full_node_df = full_node_df.drop(
    columns=["wiki_items_text"]
)

Nodes with no Wikidata: 779
Nodes with attraction_count = -1: 1240
Nodes with attraction_missing_flag = 1: 779
Same node sets: False


In [23]:
model_root = Path("/home/najla/dev/najla-msc/bikeshare/data/processed/model_df")

# Set the saved MiniLM file.
embedding_cache_file = (
    model_root
    / "cat_graph_minilm.parquet"
)

full_node_df.to_parquet(
    embedding_cache_file,
    index=False,
)

full_node_df.head(10)

,loc_id,loc_name,population,jobs,area,poi_arts_and_entertainment,poi_community_and_government,poi_cultural_and_historic,poi_education,poi_food_and_drink,...,wiki_emb_374,wiki_emb_375,wiki_emb_376,wiki_emb_377,wiki_emb_378,wiki_emb_379,wiki_emb_380,wiki_emb_381,wiki_emb_382,wiki_emb_383
0,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
1,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
2,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
3,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
4,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
5,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
6,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
7,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
8,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662
9,5350001.00,0001.00,0.000088,0.000055,6820000.0,0.000004,0.000002,5.816377e-07,7.270471e-07,0.000004,...,0.007415,-0.001172,0.018872,-0.011002,0.040623,-0.050401,0.101991,0.052261,0.003068,0.031662


In [24]:
full_node_df.columns

Index(['loc_id', 'loc_name', 'population', 'jobs', 'area',
       'poi_arts_and_entertainment', 'poi_community_and_government',
       'poi_cultural_and_historic', 'poi_education', 'poi_food_and_drink',
       ...
       'wiki_emb_374', 'wiki_emb_375', 'wiki_emb_376', 'wiki_emb_377',
       'wiki_emb_378', 'wiki_emb_379', 'wiki_emb_380', 'wiki_emb_381',
       'wiki_emb_382', 'wiki_emb_383'],
      dtype='str', length=439)

In [25]:
stop

NameError: name 'stop' is not defined

GPT embedings

In [26]:
# =========================================================
# Replace MiniLM embeddings in full_node_df with GPT embeddings
# =========================================================

import os
from getpass import getpass
from openai import OpenAI

# Ask for the key without showing it in the notebook output.
os.environ["OPENAI_API_KEY"] = getpass("Paste OpenAI API key:")

# Create OpenAI client.
# Make sure OPENAI_API_KEY is set in your environment.
client = OpenAI()

# Choose GPT embedding model.
GPT_EMBEDDING_MODEL = "text-embedding-3-large"

 #Choose GPT embedding model.
GPT_EMBEDDING_MODEL2 = "text-embedding-3-small"


In [27]:

# -----------------------------
# 1. Drop existing MiniLM columns
# -----------------------------

# Find MiniLM embedding columns already inside full_node_df.
wiki_cols = [
    col
    for col in full_node_df.columns
    if col.startswith("wiki_emb_")
]

# Drop old MiniLM embedding columns.
full_node_df = full_node_df.drop(
    columns=wiki_cols,
    errors="ignore",
)


# -----------------------------
# 2. Replace wiki_items_text
# -----------------------------


# Keep one Wikidata text row per loc_id.
wikidata_text_df = (
    full_node_wiki[["loc_id", "wiki_items_text"]]
    .drop_duplicates(subset=["loc_id"])
    .reset_index(drop=True)
)

# Left join fresh wiki_items_text from wikidata_tokens_df.
full_node_df = full_node_df.merge(
    wikidata_text_df,
    on="loc_id",
    how="left",
)

# Fill nodes without Wikidata text.
full_node_df["wiki_items_text"] = (
    full_node_df["wiki_items_text"]
    .fillna("no_wikidata")
)

# Keep one text row per loc_id for embedding.
full_node_wiki = (
    full_node_df[["loc_id", "wiki_items_text"]]
    .drop_duplicates(subset=["loc_id"])
    .reset_index(drop=True)
)

full_node_wiki.head()


,loc_id,wiki_items_text
0,5350001.00,parks_nature_and_recreation generic_buildings_...
1,5350003.00,generic_buildings_and_structures retail_and_co...
2,5350004.00,generic_buildings_and_structures culture_and_p...
3,5350005.00,religion_and_worship settlement_and_districts ...
4,5350007.01,no_wikidata


In [28]:
full_node_wiki.shape

(1247, 2)

In [29]:
# -----------------------------
# 3. Create GPT embeddings
# -----------------------------

def create_gpt_embeddings(text_list, model_name, batch_size=100):
    # Store all embedding vectors.
    all_embeddings = []

    # Loop over the text in batches.
    for start in range(0, len(text_list), batch_size):
        # Select one batch.
        batch_text = text_list[start:start + batch_size]

        # Request GPT embeddings for this batch.
        response = client.embeddings.create(
            model=model_name,
            input=batch_text,
        )

        # Extract vectors in the same order as input text.
        batch_embeddings = [
            item.embedding
            for item in response.data
        ]

        # Store this batch.
        all_embeddings.extend(batch_embeddings)

        # Print progress.
        print(f"Embedded {min(start + batch_size, len(text_list))} / {len(text_list)}")

    # Return as NumPy array.
    return np.asarray(all_embeddings, dtype=np.float32)


# Create GPT embeddings from wiki_items_text.
gpt_embeddings = create_gpt_embeddings(
    text_list=full_node_wiki["wiki_items_text"].tolist(),
    model_name=GPT_EMBEDDING_MODEL,
    batch_size=100,
)

# Name GPT embedding columns.
gpt_cols = [
    f"gpt_emb_{i}"
    for i in range(gpt_embeddings.shape[1])
]

# Create GPT embedding dataframe.
gpt_embedding_df = pd.concat(
    [
        full_node_wiki[["loc_id"]].reset_index(drop=True),
        pd.DataFrame(
            gpt_embeddings,
            columns=gpt_cols,
        ),
    ],
    axis=1,
)

# Inspect GPT embeddings.
gpt_embedding_df.head()

Embedded 100 / 1247
Embedded 200 / 1247
Embedded 300 / 1247
Embedded 400 / 1247
Embedded 500 / 1247
Embedded 600 / 1247
Embedded 700 / 1247
Embedded 800 / 1247
Embedded 900 / 1247
Embedded 1000 / 1247
Embedded 1100 / 1247
Embedded 1200 / 1247
Embedded 1247 / 1247


,loc_id,gpt_emb_0,gpt_emb_1,gpt_emb_2,gpt_emb_3,gpt_emb_4,gpt_emb_5,gpt_emb_6,gpt_emb_7,gpt_emb_8,...,gpt_emb_3062,gpt_emb_3063,gpt_emb_3064,gpt_emb_3065,gpt_emb_3066,gpt_emb_3067,gpt_emb_3068,gpt_emb_3069,gpt_emb_3070,gpt_emb_3071
0,5350001.00,0.015640,0.015228,-0.015129,0.002386,0.007187,-0.013481,-0.025009,-0.013962,-0.007957,...,-0.020233,0.002758,0.006699,0.013954,-0.019119,0.041534,0.004757,0.007889,-0.023071,-0.016327
1,5350003.00,0.014816,0.014328,-0.029282,-0.000331,-0.008484,0.010330,-0.003723,-0.004906,-0.009361,...,-0.002617,-0.012535,0.004208,0.014671,-0.019821,0.042389,0.007458,-0.000837,-0.011536,-0.022491
2,5350004.00,0.006474,0.017075,-0.030884,0.000590,0.011345,-0.008209,-0.023163,-0.028259,-0.011642,...,-0.004906,0.001018,0.015617,0.007263,-0.010208,0.033966,0.010231,0.004147,-0.014923,-0.017136
3,5350005.00,0.028244,0.022903,-0.016876,0.006748,0.003172,0.004459,-0.014732,-0.019989,-0.021042,...,-0.012466,0.012001,0.002237,0.025314,-0.006668,0.041473,-0.008057,0.009918,-0.019699,-0.022659
4,5350007.01,0.019714,-0.006786,-0.017181,0.003498,0.016815,0.008141,-0.001527,0.005680,-0.016861,...,-0.019638,0.011017,0.018494,0.022736,0.002983,0.037048,-0.017471,-0.000625,-0.013229,-0.000274


In [30]:
# Create small GPT embeddings from wiki_items_text.
gpt_small_embeddings = create_gpt_embeddings(
    text_list=full_node_wiki["wiki_items_text"].tolist(),
    model_name=GPT_EMBEDDING_MODEL2,
    batch_size=100,
)

# Name the small GPT embedding columns.
gpt_small_cols = [
    f"gpt_small_emb_{i}"
    for i in range(gpt_small_embeddings.shape[1])
]

# Create the small GPT embedding dataframe.
gpt_small_embedding_df = pd.concat(
    [
        full_node_wiki[["loc_id"]].reset_index(drop=True),
        pd.DataFrame(
            gpt_small_embeddings,
            columns=gpt_small_cols,
        ),
    ],
    axis=1,
)

gpt_small_embedding_df

Embedded 100 / 1247
Embedded 200 / 1247
Embedded 300 / 1247
Embedded 400 / 1247
Embedded 500 / 1247
Embedded 600 / 1247
Embedded 700 / 1247
Embedded 800 / 1247
Embedded 900 / 1247
Embedded 1000 / 1247
Embedded 1100 / 1247
Embedded 1200 / 1247
Embedded 1247 / 1247


,loc_id,gpt_small_emb_0,gpt_small_emb_1,gpt_small_emb_2,gpt_small_emb_3,gpt_small_emb_4,gpt_small_emb_5,gpt_small_emb_6,gpt_small_emb_7,gpt_small_emb_8,...,gpt_small_emb_1526,gpt_small_emb_1527,gpt_small_emb_1528,gpt_small_emb_1529,gpt_small_emb_1530,gpt_small_emb_1531,gpt_small_emb_1532,gpt_small_emb_1533,gpt_small_emb_1534,gpt_small_emb_1535
0,5350001.00,0.011063,-0.040894,0.044464,0.023865,-0.026581,-0.002302,-0.015404,-0.025177,-0.010201,...,-0.001081,0.062225,0.005409,-0.035400,-0.016754,-0.008614,0.009727,-0.015839,-0.004707,0.002600
1,5350003.00,0.004612,0.011002,0.027603,-0.002634,-0.027115,0.005657,-0.012024,-0.023499,-0.004097,...,0.021500,0.045685,0.016098,-0.027893,-0.021896,-0.030731,0.007553,-0.023026,0.008224,0.007538
2,5350004.00,0.015205,0.010307,0.047455,0.021393,-0.007435,0.001671,-0.013435,-0.021790,-0.028976,...,0.027069,0.031342,-0.020767,-0.041534,-0.031067,-0.026215,0.009354,-0.017502,0.017075,-0.022369
3,5350005.00,-0.006844,-0.029129,0.045197,0.018875,-0.018005,0.030212,-0.025009,0.010025,-0.000291,...,0.013474,0.038727,0.005642,-0.019730,-0.029648,0.000520,-0.017517,-0.006638,-0.000448,-0.002266
4,5350007.01,-0.024963,0.035675,0.008354,-0.016785,-0.044189,-0.022568,-0.047424,-0.029648,-0.059326,...,-0.011154,0.042633,0.006561,-0.028885,0.005112,0.021759,-0.006622,0.012512,0.027664,0.010941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1242,5680101.00,-0.024979,0.035645,0.008339,-0.016769,-0.044189,-0.022507,-0.047455,-0.029633,-0.059357,...,-0.011147,0.042633,0.006554,-0.028900,0.005119,0.021759,-0.006618,0.012512,0.027649,0.010933
1243,5680102.01,-0.024979,0.035645,0.008339,-0.016769,-0.044189,-0.022507,-0.047455,-0.029633,-0.059357,...,-0.011147,0.042633,0.006554,-0.028900,0.005119,0.021759,-0.006618,0.012512,0.027649,0.010933
1244,5680103.05,-0.024979,0.035645,0.008339,-0.016769,-0.044189,-0.022507,-0.047455,-0.029633,-0.059357,...,-0.011147,0.042633,0.006554,-0.028900,0.005119,0.021759,-0.006618,0.012512,0.027649,0.010933
1245,5680103.08,-0.024979,0.035645,0.008339,-0.016769,-0.044189,-0.022507,-0.047455,-0.029633,-0.059357,...,-0.011147,0.042633,0.006554,-0.028900,0.005119,0.021759,-0.006618,0.012512,0.027649,0.010933


In [31]:
# Set the saved MiniLM file.
gpt_cache_file = (
    model_root
    / "cat_graph_gpt.parquet"
)

gpt_embedding_df.to_parquet(gpt_cache_file,index=False,)

In [32]:
gpt_small_cache_file = (
    model_root
    / "cat_graph_gpt_small.parquet"
)

gpt_small_embedding_df.to_parquet(
    gpt_small_cache_file,
    index=False,
)

In [33]:
stop

NameError: name 'stop' is not defined

Old code => to be deleted

In [ ]:

# Read the MiniLM text embeddings into a dataframe.
text_embeddings_df = pd.read_parquet(data_dir / "processed/graph/all_wikidata_minilm_embeddings.parquet")

# Display the first few rows to inspect the data.
text_embeddings_df.head()

In [ ]:
full_node_df = (
    full_node_df.merge(
        text_embeddings_df,
        on="loc_id",
        how="left",
    )
    .drop(
        columns="wiki_items_text"
    )
)

full_node_df.head()

In [ ]:
# to be used later with SPCV
labelled_df = full_node_df[
    full_node_df["inflow_count"].notna()
    | full_node_df["outflow_count"].notna()
].copy()
labelled_df

In [ ]:
error

In [ ]:
"""
how the embedings of all text should be handled?
re-produce the embeddings for all nodes with nodes no embedings as "no_wikidate"?
"""

In [ ]:
# Main columns
date_col = "end_date"
id_col = "end_loc_id"
target_col = "trip_count"
group_col = "spatial_group"
transformed_target_col = "y_log"


# Categorical input features
categorical_cols = [
    "day_type",
    "public_holiday",
    "Main_Weather_Category",
    "season",
]


# MiniLM embedding columns
wiki_cols = [
    col
    for col in full_node_df.columns
    if col.startswith("wiki_emb_")
]


# Columns that must not be used as model features
drop_feature_cols = [
    date_col,
    id_col,
    target_col,
    group_col,
]


# Numerical input features
numerical_cols = [
    col
    for col in full_node_df.columns
    if (
        col not in drop_feature_cols
        and col not in categorical_cols
        and col not in wiki_cols
    )
]


print("Numerical features:", len(numerical_cols))
print("Categorical features:", len(categorical_cols))
print("MiniLM features:", len(wiki_cols))

In [ ]:
# =========================================================
# 3. Create 10 spatial cross-validation folds
# =========================================================

spcv_folds = mh.split_by_spatial_group_cv(
    df=labelled_df,
    group_col="spatial_group",
    n_splits=5,
    seed=42,
)

print("Number of SPCV folds:", len(spcv_folds))

for fold in spcv_folds:
    print("=" * 60)
    print("Fold:", fold["fold_id"])

    print("Train CTs:", fold["train_df"]["end_loc_id"].nunique())
    print("Validation CTs:", fold["val_df"]["end_loc_id"].nunique())
    print("Test CTs:", fold["test_df"]["end_loc_id"].nunique())

    print("Train groups:", fold["train_groups"])
    print("Validation groups:", fold["val_groups"])
    print("Test groups:", fold["test_groups"])

Helpers 

In [ ]:
# =========================================================
# Build feature preprocessor
# REUSABLE FOR GAT
# =========================================================

def build_feature_preprocessor(
    numerical_cols,
    categorical_cols,
):
    """
    Standardise numerical features and one-hot encode
    categorical features.
    """

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                StandardScaler(),
                numerical_cols,
            ),
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
                categorical_cols,
            ),
        ],
        remainder="drop",
    )

    return preprocessor

In [ ]:
# =========================================================
# 4. Prepare one spatial CV fold for GAT
# log target, PCA text, scale numerical, one-hot categorical
# =========================================================

def prepare_gat_fold(
    fold,
    numerical_cols,
    categorical_cols,
    wiki_cols,
    n_wiki_pca=16,
):
    train_df = fold["train_df"].copy()
    val_df = fold["val_df"].copy()
    test_df = fold["test_df"].copy()

    # Reuse model_helper target transform
    train_df, val_df, test_df = mh.add_target_transform(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        target_col=target_col,
        transformed_col=transformed_target_col,
        transform="log1p",
    )

    # Fit PCA only on training text embeddings
    pca = PCA(
        n_components=n_wiki_pca,
        random_state=42,
    )

    wiki_pca_cols = [
        f"wiki_pca_{i}"
        for i in range(n_wiki_pca)
    ]

    train_df[wiki_pca_cols] = pca.fit_transform(train_df[wiki_cols].fillna(0))
    val_df[wiki_pca_cols] = pca.transform(val_df[wiki_cols].fillna(0))
    test_df[wiki_pca_cols] = pca.transform(test_df[wiki_cols].fillna(0))

    # Add PCA text features to numerical features
    fold_numerical_cols = numerical_cols + wiki_pca_cols

    # Reuse the same MLP preprocessor logic
    preprocessor = build_feature_preprocessor(
        numerical_cols=fold_numerical_cols,
        categorical_cols=categorical_cols,
    )

    # Fit only on training rows
    X_train = preprocessor.fit_transform(train_df)
    X_val = preprocessor.transform(val_df)
    X_test = preprocessor.transform(test_df)

    X_train = np.asarray(X_train, dtype=np.float32)
    X_val = np.asarray(X_val, dtype=np.float32)
    X_test = np.asarray(X_test, dtype=np.float32)

    # Extract transformed targets
    y_train = train_df[transformed_target_col].to_numpy(dtype=np.float32)
    y_val = val_df[transformed_target_col].to_numpy(dtype=np.float32)
    y_test = test_df[transformed_target_col].to_numpy(dtype=np.float32)

    return {
        "fold_id": fold["fold_id"],
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
        "input_dim": X_train.shape[1],
        "train_df": train_df,
        "val_df": val_df,
        "test_df": test_df,
        "preprocessor": preprocessor,
        "pca": pca,
        "train_groups": fold["train_groups"],
        "val_groups": fold["val_groups"],
        "test_groups": fold["test_groups"],
    }

Running the functions

In [ ]:
# prep the folds
gat_folds = []

for fold in spcv_folds:
    gat_fold = prepare_gat_fold(
        fold=fold,
        numerical_cols=numerical_cols,
        categorical_cols=categorical_cols,
        wiki_cols=wiki_cols,
    )

    gat_folds.append(gat_fold)

print("Prepared folds:", len(gat_folds))

In [ ]:
gat_folds[0]

In [ ]:
# =========================================================
# Prepare GAT graph snapshots
# one graph snapshot = one date containing all CT nodes
# =========================================================

from torch_geometric.data import Data  # Import PyTorch Geometric graph container.

feature_cols = [  # Create names for the processed feature columns from X_train/X_val/X_test.
    f"x_{i}"  # Name each processed feature as x_0, x_1, x_2, ...
    for i in range(gat_fold["input_dim"])  # Use the number of processed features from step 4.
]
feature_cols

In [ ]:
# =========================================================
# Attach processed features back to metadata
# =========================================================

metadata_cols = [  # Keep only columns needed to rebuild CT-date graph rows.
    date_col,  # Date of the observation.
    id_col,  # CT identifier.
    target_col,  # Original trip_count target.
    transformed_target_col,  # Log-transformed target y_log.
    group_col,  # Spatial CV group.
]


def attach_features(split_df, X, split_name):  # Define helper to attach processed X back to metadata.
    """
    Input:
        split_df: train/val/test dataframe before feature processing.
        X: processed feature matrix for this split.
        split_name: train, val, or test.

    Output:
        dataframe containing metadata + processed features + split label.

    Description:
        Keeps target availability separate from split assignment.
    """

    meta = split_df[metadata_cols].copy().reset_index(drop=True)  # Keep metadata and reset row order.

    features = pd.DataFrame(X, columns=feature_cols)  # Convert processed features to DataFrame.

    out = pd.concat([meta, features], axis=1)  # Combine metadata and processed features.

    out["split"] = split_name  # Record whether row belongs to train, val, or test.

    out["has_label"] = (  # Create Boolean flag for observed target availability.
        out[target_col].notna()  # Original target must exist.
        & out[transformed_target_col].notna()  # Transformed target must exist.
    )

    return out  # Return processed CT-date rows.


train_rows = attach_features(gat_fold["train_df"], gat_fold["X_train"], "train")  # Build training rows.

val_rows = attach_features(gat_fold["val_df"], gat_fold["X_val"], "val")  # Build validation rows.

test_rows = attach_features(gat_fold["test_df"], gat_fold["X_test"], "test")  # Build test rows.

gat_rows = pd.concat(  # Combine all splits for snapshot construction.
    [train_rows, val_rows, test_rows],
    axis=0,
    ignore_index=True,
)

print("gat_rows shape:", gat_rows.shape)  # Print combined row count.

print("Unique dates in gat_rows:", gat_rows[date_col].nunique())  # Print number of graph snapshots expected.

print("Rows with observed labels:", gat_rows["has_label"].sum())  # Print labelled rows only.

print("Unique labelled CTs:", gat_rows.loc[gat_rows["has_label"], id_col].nunique())  # Print labelled CT count.

In [ ]:
train_rows

In [ ]:
gat_rows

In [ ]:
# =========================================================
# Prepare edge weights
# =========================================================

edges_for_gat = edges.copy()  # Copy fixed Toronto graph edges.

edges_for_gat["edge_weight"] = 1 / (edges_for_gat["weight"] + 1e-6)  # Convert distance to inverse-distance affinity.

edges_for_gat.head()  # Display edge table sample.

In [ ]:
# =========================================================
# Rename edge endpoint columns
# =========================================================

edges_for_gat = edges_for_gat.rename(  # Rename edge endpoint columns.
    columns={
        "level_0": "src_idx",  # Source CT ID.
        "level_1": "dst_idx",  # Destination CT ID.
    }
)

edges_for_gat.head()  # Display renamed edge table.

In [ ]:
"""
the spatial graph must be undirected because if CT 0 is adjacent to CT 4, CT 4 is also adjacent to CT 0.
"""
forward_edges = edges_for_gat[  # Keep original edge direction.
    ["src_idx", "dst_idx", "edge_weight"]
].copy()

reverse_edges = forward_edges.rename(  # Create reverse edge direction.
    columns={
        "src_idx": "dst_idx",  # Swap source into destination.
        "dst_idx": "src_idx",  # Swap destination into source.
    }
)

edge_df = pd.concat(  # Combine forward and reverse edges.
    [forward_edges, reverse_edges],
    ignore_index=True,
)

edge_df = edge_df.drop_duplicates(  # Remove duplicated directed edges.
    ["src_idx", "dst_idx"]
).reset_index(drop=True)

print("Directed PyG edges:", len(edge_df))  # Print number of directed edges.

edge_df  # Display edge dataframe.

In [ ]:
edge_df.drop_duplicates()

In [ ]:
# =========================================================
# Create FULL graph node index
# Full graph nodes come from Toronto CT graph edges and all_node_features
# NOT from labelled gat_rows
# =========================================================

graph_node_ids = sorted(
    set(edge_df["src_idx"].dropna().astype(str))
    | set(edge_df["dst_idx"].dropna().astype(str))
)

node_feature_ids = sorted(
    all_node_features[id_col].dropna().astype(str).unique()
)

node_ids = sorted(
    set(graph_node_ids)
    | set(node_feature_ids)
)

node_to_idx = {
    node_id: idx
    for idx, node_id in enumerate(node_ids)
}

print("Full graph nodes:", len(node_ids))
print("Nodes from graph edges:", len(graph_node_ids))
print("Nodes from all_node_features:", len(node_feature_ids))
print("Labelled nodes in gat_rows:", gat_rows[id_col].nunique())

list(node_to_idx.items())[:10]

In [ ]:
"""
index are needed for both the nodes and the edges, MUST be integers
"""
# =========================================================
# Map graph edges from CT IDs to integer node indices
# =========================================================

edge_df_for_tensor = edge_df.copy()  # Copy edge dataframe.

edge_df_for_tensor["src_idx"] = (  # Convert source CT IDs to integer node indices.
    edge_df_for_tensor["src_idx"].astype(str).map(node_to_idx)
)

edge_df_for_tensor["dst_idx"] = (  # Convert destination CT IDs to integer node indices.
    edge_df_for_tensor["dst_idx"].astype(str).map(node_to_idx)
)

edge_df_for_tensor = edge_df_for_tensor.dropna(  # Keep only edges where both endpoints exist in node_to_idx.
    subset=["src_idx", "dst_idx"]
).copy()

edge_df_for_tensor["src_idx"] = edge_df_for_tensor["src_idx"].astype("int64")  # Convert source to integer.

edge_df_for_tensor["dst_idx"] = edge_df_for_tensor["dst_idx"].astype("int64")  # Convert destination to integer.

print("Edges kept after mapping:", len(edge_df_for_tensor))  # Print mapped edge count.

edge_df_for_tensor # Display mapped edges.

In [ ]:
# =========================================================
# Build PyTorch Geometric edge tensors
# =========================================================

edge_index = torch.tensor(  # Build PyG edge index tensor.
    edge_df_for_tensor[["src_idx", "dst_idx"]].to_numpy().T,  # Convert to shape [2, num_edges].
    dtype=torch.long,  # PyG requires long integer indices.
)

edge_weight = torch.tensor(  # Build edge weight tensor.
    edge_df_for_tensor["edge_weight"].to_numpy(dtype=np.float32),  # Use inverse-distance affinity.
    dtype=torch.float32,  # Store as float tensor.
)

edge_attr = edge_weight.view(-1, 1)  # Reshape edge weights to [num_edges, 1] for edge attributes.

print("edge_index shape:", edge_index.shape)  # Should be [2, num_edges].

print("edge_weight shape:", edge_weight.shape)  # Should be [num_edges].

print("edge_attr shape:", edge_attr.shape)  # Should be [num_edges, 1].

In [ ]:
# =========================================================
# Debug one GAT graph snapshot
# Full graph is used, but loss masks are only True for labelled CTs
# =========================================================

debug_snapshots = []  # Create empty list for debug snapshot.

for snapshot_date, date_df in gat_rows.groupby(date_col, sort=True):  # Loop through dates.

    print("=" * 80)  # Print separator.
    print("STEP 1: Current snapshot date")  # Print step label.
    print(snapshot_date)  # Print current date.

    print("\nSTEP 2: Rows available for this date")  # Print step label.
    print("date_df shape:", date_df.shape)  # Print rows for this date.

    display(  # Show key columns for this date.
        date_df[
            [date_col, id_col, target_col, transformed_target_col, "split", "has_label"]
        ].head()
    )

    print("\nSTEP 3: Create empty tensors for FULL graph nodes")  # Print step label.

    x_t = torch.zeros(  # Create feature matrix for all full graph nodes.
        (len(node_ids), len(feature_cols)),
        dtype=torch.float32,
    )

    y_t = torch.zeros(  # Create transformed target vector for all full graph nodes.
        len(node_ids),
        dtype=torch.float32,
    )

    y_raw_t = torch.zeros(  # Create raw target vector for all full graph nodes.
        len(node_ids),
        dtype=torch.float32,
    )

    print("Full graph node count:", len(node_ids))  # Print number of full graph nodes.
    print("x_t shape:", x_t.shape)  # Print feature matrix shape.
    print("y_t shape:", y_t.shape)  # Print transformed target shape.
    print("y_raw_t shape:", y_raw_t.shape)  # Print raw target shape.

    print("\nSTEP 4: Create empty masks for FULL graph nodes")  # Print step label.

    feature_mask = torch.zeros(  # True where this date has feature rows.
        len(node_ids),
        dtype=torch.bool,
    )

    label_mask = torch.zeros(  # True where this date has observed inflow target.
        len(node_ids),
        dtype=torch.bool,
    )

    train_mask = torch.zeros(  # True for labelled training nodes.
        len(node_ids),
        dtype=torch.bool,
    )

    val_mask = torch.zeros(  # True for labelled validation nodes.
        len(node_ids),
        dtype=torch.bool,
    )

    test_mask = torch.zeros(  # True for labelled test nodes.
        len(node_ids),
        dtype=torch.bool,
    )

    print("feature_mask true count:", feature_mask.sum().item())  # Print initial feature count.
    print("label_mask true count:", label_mask.sum().item())  # Print initial label count.
    print("train_mask true count:", train_mask.sum().item())  # Print initial train count.
    print("val_mask true count:", val_mask.sum().item())  # Print initial validation count.
    print("test_mask true count:", test_mask.sum().item())  # Print initial test count.

    print("\nSTEP 5: Convert CT IDs to full-graph node indices")  # Print step label.

    node_idx = date_df[id_col].astype(str).map(node_to_idx)  # Map CT IDs to full graph node indices.

    print("First 10 CT IDs:")  # Print label.
    print(date_df[id_col].head(10).to_list())  # Print first CT IDs.

    print("First 10 mapped node indices:")  # Print label.
    print(node_idx.head(10).to_list())  # Print first mapped indices.

    print("\nSTEP 6: Keep only rows whose CT exists in the full graph")  # Print step label.

    keep = node_idx.notna()  # Keep rows with valid graph node index.

    print("Rows before keep:", len(date_df))  # Print rows before filtering.
    print("Rows after keep:", keep.sum())  # Print rows after filtering.

    print("\nSTEP 7: Fill x_t for nodes with available features")  # Print step label.

    feature_idx = torch.tensor(  # Convert kept node indices to torch tensor.
        node_idx[keep].to_numpy(dtype=np.int64),
        dtype=torch.long,
    )

    x_t[feature_idx] = torch.tensor(  # Fill feature matrix for available CT rows.
        date_df.loc[keep, feature_cols].to_numpy(dtype=np.float32),
        dtype=torch.float32,
    )

    feature_mask[feature_idx] = True  # Mark nodes with available features for this date.

    print("feature_idx shape:", feature_idx.shape)  # Print feature index shape.
    print("feature_mask true count:", feature_mask.sum().item())  # Print number of nodes with features.

    print("First 3 filled feature node indices:")  # Print label.
    print(feature_idx[:3])  # Print first filled node indices.

    print("x_t at first 3 filled nodes:")  # Print label.
    print(x_t[feature_idx[:3]])  # Print first filled feature rows.

    print("\nSTEP 8: Identify labelled rows only")  # Print step label.

    label_rows = keep & date_df["has_label"]  # Keep rows that exist in graph and have observed target.

    print("Rows with features:", keep.sum())  # Print rows with features.
    print("Rows with observed labels:", label_rows.sum())  # Print rows with labels.

    label_idx = torch.tensor(  # Convert labelled node indices to tensor.
        node_idx[label_rows].to_numpy(dtype=np.int64),
        dtype=torch.long,
    )

    print("label_idx shape:", label_idx.shape)  # Print labelled index shape.
    print("First 10 labelled node indices:")  # Print label.
    print(label_idx[:10])  # Print first labelled node indices.

    print("\nSTEP 9: Fill y_t and y_raw_t for labelled nodes only")  # Print step label.

    y_t[label_idx] = torch.tensor(  # Fill transformed target for labelled nodes.
        date_df.loc[label_rows, transformed_target_col].to_numpy(dtype=np.float32),
        dtype=torch.float32,
    )

    y_raw_t[label_idx] = torch.tensor(  # Fill raw target for labelled nodes.
        date_df.loc[label_rows, target_col].to_numpy(dtype=np.float32),
        dtype=torch.float32,
    )

    label_mask[label_idx] = True  # Mark observed-label nodes.

    print("label_mask true count:", label_mask.sum().item())  # Print labelled node count.

    print("y_t at first 10 labelled nodes:")  # Print label.
    print(y_t[label_idx[:10]])  # Print transformed target values.

    print("y_raw_t at first 10 labelled nodes:")  # Print label.
    print(y_raw_t[label_idx[:10]])  # Print raw target values.

    print("\nSTEP 10: Read split labels for labelled nodes")  # Print step label.

    split_values = date_df.loc[label_rows, "split"]  # Get split labels only for labelled rows.

    print(split_values.value_counts())  # Print split distribution among labelled rows.

    print("\nSTEP 11: Fill train/val/test masks for labelled nodes only")  # Print step label.

    train_mask[label_idx] = torch.tensor(  # Mark labelled training nodes.
        split_values.eq("train").to_numpy(),
        dtype=torch.bool,
    )

    val_mask[label_idx] = torch.tensor(  # Mark labelled validation nodes.
        split_values.eq("val").to_numpy(),
        dtype=torch.bool,
    )

    test_mask[label_idx] = torch.tensor(  # Mark labelled test nodes.
        split_values.eq("test").to_numpy(),
        dtype=torch.bool,
    )

    print("feature_mask true count:", feature_mask.sum().item())  # Print feature nodes.
    print("label_mask true count:", label_mask.sum().item())  # Print labelled nodes.
    print("train_mask true count:", train_mask.sum().item())  # Print training labelled nodes.
    print("val_mask true count:", val_mask.sum().item())  # Print validation labelled nodes.
    print("test_mask true count:", test_mask.sum().item())  # Print test labelled nodes.

    print("\nSTEP 12: Sanity checks")  # Print step label.

    assert train_mask.sum() + val_mask.sum() + test_mask.sum() == label_mask.sum(), (
        "Every labelled node should belong to exactly one split."
    )  # Check labelled nodes are assigned to one split.

    assert torch.all(train_mask <= label_mask), "train_mask must be subset of label_mask."  # Check train subset.

    assert torch.all(val_mask <= label_mask), "val_mask must be subset of label_mask."  # Check val subset.

    assert torch.all(test_mask <= label_mask), "test_mask must be subset of label_mask."  # Check test subset.

    print("Sanity checks passed.")  # Confirm checks.

    print("\nSTEP 13: Create PyG Data snapshot")  # Print step label.

    snapshot = Data(  # Create one PyTorch Geometric graph snapshot.
        x=x_t,  # Node feature matrix for full graph nodes.
        y=y_t,  # Log-transformed target vector.
        y_raw=y_raw_t,  # Raw trip_count target vector.
        edge_index=edge_index,  # Fixed Toronto graph edges.
        edge_weight=edge_weight,  # Fixed inverse-distance edge weights.
        edge_attr=edge_attr,  # Edge attributes for GAT layer if edge_dim is used.
        feature_mask=feature_mask,  # Nodes with feature rows on this date.
        label_mask=label_mask,  # Nodes with observed target.
        train_mask=train_mask,  # Labelled training nodes.
        val_mask=val_mask,  # Labelled validation nodes.
        test_mask=test_mask,  # Labelled test nodes.
        snapshot_date=snapshot_date,  # Date represented by this snapshot.
        num_nodes=len(node_ids),  # Explicit full graph node count.
    )

    debug_snapshots.append(snapshot)  # Store debug snapshot.

    print(snapshot)  # Print PyG Data object.

    print("\nDone debugging one date.")  # Print completion message.

    break  # Stop after one date for debugging only.

In [ ]:
# =========================================================
# Real version: prepare GAT graph snapshots
# Full Toronto graph is used.
# Labelled rows only are used for x/y/masks.
# Loss is calculated only on labelled train/val/test masks.
# =========================================================

gat_snapshots = []


for snapshot_date, date_df in gat_rows.groupby(date_col, sort=True):

    # -----------------------------------------------------
    # 1. Create full-size tensors for all Toronto CT nodes
    # -----------------------------------------------------

    x_t = torch.zeros(
        (len(node_ids), len(feature_cols)),
        dtype=torch.float32,
    )

    y_t = torch.zeros(
        len(node_ids),
        dtype=torch.float32,
    )

    y_raw_t = torch.zeros(
        len(node_ids),
        dtype=torch.float32,
    )

    # -----------------------------------------------------
    # 2. Create full-size masks
    # -----------------------------------------------------

    feature_mask = torch.zeros(
        len(node_ids),
        dtype=torch.bool,
    )

    label_mask = torch.zeros(
        len(node_ids),
        dtype=torch.bool,
    )

    train_mask = torch.zeros(
        len(node_ids),
        dtype=torch.bool,
    )

    val_mask = torch.zeros(
        len(node_ids),
        dtype=torch.bool,
    )

    test_mask = torch.zeros(
        len(node_ids),
        dtype=torch.bool,
    )

    # -----------------------------------------------------
    # 3. Map labelled CT rows to full-graph node positions
    # -----------------------------------------------------

    node_idx = (
        date_df[id_col]
        .astype(str)
        .map(node_to_idx)
    )

    keep = node_idx.notna()

    idx = torch.tensor(
        node_idx[keep].to_numpy(dtype=np.int64),
        dtype=torch.long,
    )

    # -----------------------------------------------------
    # 4. Fill x only for labelled processed rows
    # -----------------------------------------------------

    x_t[idx] = torch.tensor(
        date_df.loc[keep, feature_cols].to_numpy(dtype=np.float32),
        dtype=torch.float32,
    )

    feature_mask[idx] = True

    # -----------------------------------------------------
    # 5. Fill y and masks only for rows with observed labels
    # -----------------------------------------------------

    label_rows = keep & date_df["has_label"]

    label_idx = torch.tensor(
        node_idx[label_rows].to_numpy(dtype=np.int64),
        dtype=torch.long,
    )

    y_t[label_idx] = torch.tensor(
        date_df.loc[label_rows, transformed_target_col].to_numpy(dtype=np.float32),
        dtype=torch.float32,
    )

    y_raw_t[label_idx] = torch.tensor(
        date_df.loc[label_rows, target_col].to_numpy(dtype=np.float32),
        dtype=torch.float32,
    )

    label_mask[label_idx] = True

    split_values = date_df.loc[label_rows, "split"]

    train_mask[label_idx] = torch.tensor(
        split_values.eq("train").to_numpy(),
        dtype=torch.bool,
    )

    val_mask[label_idx] = torch.tensor(
        split_values.eq("val").to_numpy(),
        dtype=torch.bool,
    )

    test_mask[label_idx] = torch.tensor(
        split_values.eq("test").to_numpy(),
        dtype=torch.bool,
    )

    # -----------------------------------------------------
    # 6. Sanity checks
    # -----------------------------------------------------

    assert torch.all(train_mask <= label_mask), (
        "train_mask must be a subset of label_mask."
    )

    assert torch.all(val_mask <= label_mask), (
        "val_mask must be a subset of label_mask."
    )

    assert torch.all(test_mask <= label_mask), (
        "test_mask must be a subset of label_mask."
    )

    assert (
        train_mask.sum()
        + val_mask.sum()
        + test_mask.sum()
    ) == label_mask.sum(), (
        "Every labelled node should belong to exactly one split."
    )

    # -----------------------------------------------------
    # 7. Create one full-graph PyG snapshot
    # -----------------------------------------------------

    snapshot = Data(
        x=x_t,
        y=y_t,
        y_raw=y_raw_t,
        edge_index=edge_index,
        edge_weight=edge_weight,
        edge_attr=edge_attr,
        feature_mask=feature_mask,
        label_mask=label_mask,
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask,
        snapshot_date=snapshot_date,
        num_nodes=len(node_ids),
    )

    gat_snapshots.append(snapshot)

print("Number of snapshots:", len(gat_snapshots))

gat_snapshots[0]

In [ ]:
data = gat_snapshots[0]

print("Snapshot date:", data.snapshot_date)

print("Number of nodes:", data.num_nodes)
print("Number of features:", data.x.shape[1])
print("Number of directed edges:", data.edge_index.shape[1])

print("feature_mask true count:", data.feature_mask.sum().item())
print("label_mask true count:", data.label_mask.sum().item())
print("train_mask true count:", data.train_mask.sum().item())
print("val_mask true count:", data.val_mask.sum().item())
print("test_mask true count:", data.test_mask.sum().item())

print(
    "train + val + test:",
    (
        data.train_mask.sum()
        + data.val_mask.sum()
        + data.test_mask.sum()
    ).item()
)

print("label_mask:", data.label_mask.sum().item())

In [ ]:
print("train_df rows:", gat_fold["train_df"].shape)
print("val_df rows:", gat_fold["val_df"].shape)
print("test_df rows:", gat_fold["test_df"].shape)

print("train unique CTs:", gat_fold["train_df"][id_col].nunique())
print("val unique CTs:", gat_fold["val_df"][id_col].nunique())
print("test unique CTs:", gat_fold["test_df"][id_col].nunique())

print("train unique dates:", gat_fold["train_df"][date_col].nunique())
print("val unique dates:", gat_fold["val_df"][date_col].nunique())
print("test unique dates:", gat_fold["test_df"][date_col].nunique())

In [ ]:
# =========================================================
# Check how many CTs are in the test_df for the current fold
# =========================================================

test_ct_count = gat_fold["test_df"][id_col].nunique()

test_row_count = len(gat_fold["test_df"])

print("Test rows:", test_row_count)
print("Unique test CTs:", test_ct_count)

In [ ]:
# =========================================================
# SPCV fold CT-count summary table
# =========================================================

fold_size_summary = []

for fold in spcv_folds:
    fold_size_summary.append(
        {
            "fold_id": fold["fold_id"],
            "train_cts": fold["train_df"][id_col].nunique(),
            "val_cts": fold["val_df"][id_col].nunique(),
            "test_cts": fold["test_df"][id_col].nunique(),
            "train_rows": len(fold["train_df"]),
            "val_rows": len(fold["val_df"]),
            "test_rows": len(fold["test_df"]),
            "train_groups": fold["train_groups"],
            "val_groups": fold["val_groups"],
            "test_groups": fold["test_groups"],
        }
    )

fold_size_summary_df = pd.DataFrame(fold_size_summary)

display(fold_size_summary_df)

In [ ]:
# =========================================================
# Check labelled CT count per spatial group
# =========================================================

group_ct_summary = (
    labelled_df
    .groupby(group_col)
    .agg(
        labelled_cts=(id_col, "nunique"),
        labelled_rows=(id_col, "size"),
        dates=(date_col, "nunique"),
    )
    .reset_index()
    .sort_values("labelled_cts")
)

display(group_ct_summary)

print("Total labelled CTs:", labelled_df[id_col].nunique())